# Custom Competitor Guard with Guardrails

This notebook demonstrates how to build a **custom Guardrails validator** that detects competitor names in user input and redacts them using `Guard().use()`.

## What you'll learn
- How to create a custom validator with `@register_validator`
- How to return `PassResult` / `FailResult` from `_validate`
- How to wrap a validator with `Guard().use()`
- How to read `ValidationOutcome` (`validation_passed`, `validated_output`, `error`)

## Step 1 — Install dependencies (run once)

If you haven't installed Guardrails yet:

```bash
uv add guardrails-ai
# or: pip install guardrails-ai
```

## Step 2 — Import required modules

In [ ]:
from guardrails.validator_base import Validator, register_validator
from guardrails_ai.types import FailResult, PassResult
from guardrails import Guard

: 

## Step 3 — Define the custom `CompetitorCheck` validator

- Registers a validator named `competition_check` for string data
- Scans input for competitor names (case-insensitive)
- Replaces matches with `[competitor]`
- Returns `FailResult` when a competitor is found, otherwise `PassResult`

In [ ]:
@register_validator(
    name="competition_check",
    data_type="string",
)
class CompetitorCheck(Validator):

    def __init__(
        self,
        competitors: list[str],
        on_fail: str | None = None,
        **kwargs,
    ):
        super().__init__(on_fail=on_fail, **kwargs)
        self.competitors = [c.lower() for c in competitors]

    def _validate(self, value, metadata):
        fixed_text = value

        for competitor in self.competitors:
            if competitor in fixed_text.lower():
                fixed_text = fixed_text.replace(competitor, "[competitor]")

        if fixed_text != value:
            return FailResult(
                error_message="Competitor detected",
                validated_chunk=fixed_text,
            )

        return PassResult(validated_chunk=value)

## Step 4 — Create the Guard

Wrap the validator with `Guard().use()`.

- `on_fail="fix"` tells Guardrails to apply the corrected value from the validator when validation fails
- Competitors list can be updated as needed

In [ ]:
COMPETITORS = ["openai", "anthropic", "google", "gemini", "microsoft"]

guard = Guard().use(
    CompetitorCheck(
        competitors=COMPETITORS,
        on_fail="fix",
    )
)

## Step 5 — Test: message containing a competitor

This message includes `openai`, so validation should fail and the competitor name should be redacted.

In [ ]:
fail_message = "My name is rahul bisht and i work at openai"

fail_outcome = guard.validate(fail_message)

print("Input:             ", fail_message)
print("validation_passed: ", fail_outcome.validation_passed)
print("validated_output:  ", fail_outcome.validated_output)
print("error:             ", fail_outcome.error)
print("full outcome:      ", fail_outcome)

## Step 6 — Test: clean message (no competitor)

This message should pass validation with no changes.

In [ ]:
pass_message = "My name is rahul bisht and I build AI chatbots"

pass_outcome = guard.validate(pass_message)

print("Input:             ", pass_message)
print("validation_passed: ", pass_outcome.validation_passed)
print("validated_output:  ", pass_outcome.validated_output)
print("error:             ", pass_outcome.error)
print("full outcome:      ", pass_outcome)

## Step 7 — Summary

| Field | Fail case | Pass case |
|---|---|---|
| `validation_passed` | `False` | `True` |
| `validated_output` | Text with `[competitor]` replacement | Original text unchanged |
| `error` | Optional error info from Guardrails | `None` |

### `on_fail` options
- `"noop"` — return outcome without modifying text; check `validation_passed` yourself
- `"fix"` — apply corrected value from validator
- `"exception"` — raise `ValidationError` on failure